In [15]:
import os
import sys
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    median_absolute_error
)
from xgboost import XGBRegressor


# ============================================================
# 1. تنظیمات و مسیر فایل‌ها
# ============================================================

START_TIME = time.perf_counter()

POWER_THRESHOLD = 90
N_ESTIMATORS = 300
IRC_FOLDS = 5
IRC_ROBUST_Z_THRESHOLD = 4.0
IRC_WINDOW = 7
IRC_WINDOW_FRACTION = 0.75
IRC_MIN_TARGETS = 3

if getattr(sys, "frozen", False):
    base_dir = os.path.dirname(sys.executable)
elif "__file__" in globals():
    base_dir = os.path.dirname(os.path.abspath(__file__))
else:
    base_dir = os.getcwd()

input_file = os.path.join(
    base_dir,
    "second_stage_inputs",
    "G12",
    "dsas_g12_generator_bearings_output.xlsx"
)

output_dir = os.path.join(
    base_dir,
    "outputs",
    "G12",
    "dsas_g12_generator_bearings_vibration",
    "residual_deviation_monitoring"
)

os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    "dsas_g12_generator_bearings_xgboost_predictions.xlsx"
)


# ============================================================
# 2. تعریف ستون‌ها
# ============================================================

DATE_COLUMN = "date"
POWER_COLUMN = "ACTIVE POWER"

rename_map = {
    "AssetID_9343": "ACTIVE POWER",
    "AssetID_9344": "REACTIVE POWER",
    "AssetID_9362": "GEN BRG VIB (COMP.) X",
    "AssetID_9363": "GEN BRG VIB (COMP.) Y",
    "AssetID_9364": "GEN BRG VIB (EXE SIDE) X",
    "AssetID_9365": "GEN BRG VIB (EXE SIDE) Y",
    "AssetID_9366": "EXE. BRG VIB X",
    "AssetID_9367": "EXE. BRG VIB Y",
    "AssetID_9371": "GEN.BRG.DR.TEMP(COMP.SIDE)",
    "AssetID_9372": "GEN.BRG.DR.TEMP(EXC.SIDE)",
    "AssetID_9373": "EXC.BRG.DR.TEMP.",
    "AssetID_9357": "AMBIENT TEMP"
}

features = [
    "ACTIVE POWER",
    "REACTIVE POWER",
    "GEN BRG VIB (COMP.) X",
    "GEN BRG VIB (COMP.) Y",
    "GEN BRG VIB (EXE SIDE) X",
    "GEN BRG VIB (EXE SIDE) Y",
    "EXE. BRG VIB X",
    "EXE. BRG VIB Y",
    "GEN.BRG.DR.TEMP(COMP.SIDE)",
    "GEN.BRG.DR.TEMP(EXC.SIDE)",
    "EXC.BRG.DR.TEMP.",
    "AMBIENT TEMP"
]

targets = [
    "GEN BRG VIB (COMP.) X",
    "GEN BRG VIB (COMP.) Y",
    "GEN BRG VIB (EXE SIDE) X",
    "GEN BRG VIB (EXE SIDE) Y",
    "EXE. BRG VIB X",
    "EXE. BRG VIB Y"
]


# ============================================================
# 3. توابع کمکی
# ============================================================

def make_xgb():
    """ساخت یک مدل XGBoost با تنظیمات یکسان."""

    return XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )


def replace_iqr_outliers(data, columns):
    """
    شناسایی مقادیر پرت با IQR و جایگزینی آن‌ها
    با میانگین همسایه قبلی و بعدی.

    این تابع فقط روی داده آموزشی اجرا می‌شود.
    """

    result = data.copy()
    replacement_counts = {}

    for col in columns:

        s = result[col].copy()

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            replacement_counts[col] = 0
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_mask = (
            (s < lower) |
            (s > upper)
        )

        previous_value = s.shift(1)
        next_value = s.shift(-1)

        neighbor_mean = (
            previous_value + next_value
        ) / 2

        neighbor_mean = neighbor_mean.fillna(
            previous_value
        )

        neighbor_mean = neighbor_mean.fillna(
            next_value
        )

        replace_mask = (
            outlier_mask &
            neighbor_mean.notna()
        )

        result.loc[replace_mask, col] = (
            neighbor_mean.loc[replace_mask]
        )

        replacement_counts[col] = int(
            replace_mask.sum()
        )

    return result, replacement_counts


def make_robust_z_score(residuals):
    """
    Robust Z-Score مبتنی بر Median و MAD.
    """

    residuals = pd.Series(
        residuals,
        dtype=float
    )

    median_value = residuals.median()

    mad = (
        residuals - median_value
    ).abs().median()

    if pd.isna(mad) or mad <= 1e-12:
        return pd.Series(
            np.nan,
            index=residuals.index
        )

    return (
        0.6745 *
        (residuals - median_value) /
        mad
    )


# ============================================================
# 4. خواندن داده خام
# ============================================================

print("\n[1/9] Reading input file...")

df = pd.read_excel(input_file)

if df.empty:
    raise ValueError("فایل ورودی خالی است.")

if DATE_COLUMN not in df.columns:
    raise ValueError(
        f"ستون تاریخ {DATE_COLUMN} در فایل موجود نیست."
    )

missing_columns = [
    col for col in rename_map
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"ستون‌های زیر موجود نیستند: {missing_columns}"
    )

df.rename(
    columns=rename_map,
    inplace=True
)

df[DATE_COLUMN] = pd.to_datetime(
    df[DATE_COLUMN],
    errors="coerce"
)

if df[DATE_COLUMN].isna().any():
    raise ValueError(
        "ستون تاریخ دارای مقدار نامعتبر است. "
        "ابتدا تاریخ‌های نامعتبر را اصلاح کنید."
    )

for col in features:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

# ترتیب زمانی برای IQR و تحلیل پایداری IRC مهم است.
df.sort_values(
    DATE_COLUMN,
    inplace=True
)

df.reset_index(drop=True, inplace=True)

input_row_count = len(df)

print("Input rows:", input_row_count)


# ============================================================
# 5. تعیین سه ماه اخیر
#    مرزبندی بر اساس داده خام و پیش از هرگونه حذف
# ============================================================

print("\n[2/9] Defining training and prediction periods...")

max_date = df[DATE_COLUMN].max()

cutoff_date = (
    max_date - pd.DateOffset(months=3)
)

old_mask = df[DATE_COLUMN] < cutoff_date
recent_mask = df[DATE_COLUMN] >= cutoff_date

old_indices = df.index[old_mask]
recent_indices = df.index[recent_mask]

if len(old_indices) == 0:
    raise ValueError(
        "داده‌ای قبل از سه ماه اخیر برای آموزش وجود ندارد."
    )

if len(recent_indices) == 0:
    raise ValueError(
        "داده‌ای در سه ماه اخیر برای پیش‌بینی وجود ندارد."
    )

print("Latest date:", max_date)
print("Three-month cutoff:", cutoff_date)
print("Historical rows:", len(old_indices))
print("Recent output rows:", len(recent_indices))


# ============================================================
# 6. فیلتر شرایط بهره‌برداری
#    فقط داده آموزشی با توان >= 90
# ============================================================

print("\n[3/9] Applying operational filter...")

train_df = df.loc[old_indices].copy()

before_power_filter = len(train_df)

train_df = train_df.loc[
    train_df[POWER_COLUMN] >= POWER_THRESHOLD
].copy()

removed_by_power = (
    before_power_filter - len(train_df)
)

print("Historical rows before power filter:",
      before_power_filter)

print("Rows removed by power filter:",
      removed_by_power)

print("Training rows after power filter:",
      len(train_df))

if len(train_df) < IRC_FOLDS * 10:
    raise ValueError(
        "داده آموزشی بعد از فیلتر توان برای اجرای مطمئن "
        "IRC و اعتبارسنجی پنج‌بخشی کافی نیست."
    )


# ============================================================
# 7. IQR: اصلاح مقادیر پرت در داده آموزشی
# ============================================================

print("\n[4/9] Applying IQR cleaning...")

train_df, iqr_counts = replace_iqr_outliers(
    train_df,
    features
)

for col, count in iqr_counts.items():
    print(f"IQR | {col}: {count} replacements")

total_iqr_replacements = sum(
    iqr_counts.values()
)

print("Total IQR replacements:",
      total_iqr_replacements)


# ============================================================
# 8. Isolation Forest
#    حذف ردیف‌های شناسایی‌شده به‌عنوان ناهنجار
# ============================================================

print("\n[5/9] Applying Isolation Forest...")

# میانه‌ها فقط از داده آموزشی محاسبه می‌شوند.
iso_medians = (
    train_df[features]
    .median()
    .fillna(0)
)

iso_input = (
    train_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(iso_medians)
)

iso_model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_labels = iso_model.fit_predict(
    iso_input
)

iso_keep_mask = iso_labels == 1

before_iso = len(train_df)

train_df = train_df.loc[
    iso_keep_mask
].copy()

removed_by_iso = (
    before_iso - len(train_df)
)

print("Rows removed by Isolation Forest:",
      removed_by_iso)

print("Rows remaining after Isolation Forest:",
      len(train_df))

if len(train_df) < IRC_FOLDS * 10:
    raise ValueError(
        "داده آموزشی پس از Isolation Forest برای IRC کافی نیست."
    )


# ============================================================
# 9. IRC: تشخیص ناهنجاری با Residual مدل XGBoost
#
# برای هر تارگت:
#   - حذف تارگت از ورودی‌ها
#   - تولید پیش‌بینی Out-of-Fold با KFold پنج‌بخشی
#   - محاسبه Residual
#   - محاسبه Robust Z-Score
#   - تشخیص ناهنجاری با |Robust Z| > 4
#   - بررسی پایداری در پنجره 7 رکوردی
#
# سپس ردیف‌هایی حذف می‌شوند که ناهنجاری پایدار را
# در حداقل سه تارگت نشان دهند.
# ============================================================

print("\n[6/9] Applying IRC residual cleaning...")

irc_flags = pd.DataFrame(
    False,
    index=train_df.index,
    columns=targets
)

irc_details = []

for target in targets:

    target_start = time.perf_counter()

    print(f"\nIRC target: {target}")

    # حذف تارگت از ورودی مدل
    input_features = [
        col for col in features
        if col != target
    ]

    X_all = train_df[input_features].copy()

    y_all = pd.to_numeric(
        train_df[target],
        errors="coerce"
    )

    X_all = X_all.replace(
        [np.inf, -np.inf],
        np.nan
    )

    valid_y_mask = y_all.notna()

    X = X_all.loc[valid_y_mask].copy()
    y = y_all.loc[valid_y_mask].copy()

    if len(X) < IRC_FOLDS * 10:
        raise ValueError(
            f"تعداد داده معتبر برای IRC تارگت {target} کافی نیست."
        )

    # حذف فیچرهای کاملاً خالی
    usable_features = [
        col for col in input_features
        if X[col].notna().any()
    ]

    if not usable_features:
        raise ValueError(
            f"ورودی معتبر برای IRC تارگت {target} وجود ندارد."
        )

    X = X[usable_features].copy()

    # میانه ورودی‌ها فقط از داده‌های آموزشی این مرحله
    feature_medians = X.median().fillna(0)

    X = X.fillna(feature_medians)

    # پیش‌بینی خارج از نمونه برای هر رکورد
    oof_predictions = pd.Series(
        np.nan,
        index=X.index,
        dtype=float
    )

    kfold = KFold(
        n_splits=IRC_FOLDS,
        shuffle=True,
        random_state=42
    )

    for fold_number, (fit_pos, val_pos) in enumerate(
        kfold.split(X),
        start=1
    ):

        X_fit = X.iloc[fit_pos]
        y_fit = y.iloc[fit_pos]

        X_val = X.iloc[val_pos]

        fold_model = make_xgb()

        fold_model.fit(
            X_fit,
            y_fit,
            verbose=False
        )

        fold_predictions = fold_model.predict(
            X_val
        )

        oof_predictions.loc[
            X_val.index
        ] = fold_predictions

    # Residual برای داده‌ای که در آموزش همان مدل نبوده
    residuals = y - oof_predictions

    valid_residuals = residuals.dropna()

    if len(valid_residuals) < IRC_FOLDS * 10:
        raise ValueError(
            f"Residual معتبر برای تارگت {target} کافی نیست."
        )

    robust_z = make_robust_z_score(
        valid_residuals
    )

    raw_anomaly = (
        robust_z.abs() >
        IRC_ROBUST_Z_THRESHOLD
    )

    raw_anomaly = raw_anomaly.fillna(False)

    # شاخص ناهنجاری در ترتیب زمانی داده‌های آموزشی
    raw_anomaly = raw_anomaly.reindex(
        train_df.index,
        fill_value=False
    )

    # در پنجره 7 رکوردی، حداقل 75 درصد رکوردها
    # باید ناهنجار باشند تا وضعیت پایدار تلقی شود.
    rolling_fraction = (
        raw_anomaly.astype(float)
        .rolling(
            window=IRC_WINDOW,
            center=True,
            min_periods=IRC_WINDOW
        )
        .mean()
    )

    persistent_anomaly = (
        raw_anomaly &
        (
            rolling_fraction >=
            IRC_WINDOW_FRACTION
        )
    )

    persistent_anomaly = persistent_anomaly.fillna(
        False
    )

    irc_flags.loc[
        persistent_anomaly.index,
        target
    ] = persistent_anomaly.astype(bool)

    raw_count = int(raw_anomaly.sum())
    persistent_count = int(persistent_anomaly.sum())

    irc_details.append({
        "Target": target,
        "Valid OOF residuals": len(valid_residuals),
        "Raw residual anomalies": raw_count,
        "Persistent anomalies": persistent_count,
        "IRC time (sec)": round(
            time.perf_counter() - target_start,
            2
        )
    })

    print("Valid OOF residuals:", len(valid_residuals))
    print("Raw residual anomalies:", raw_count)
    print("Persistent anomalies:", persistent_count)


# تعداد تارگت‌هایی که برای هر ردیف ناهنجاری پایدار دارند
irc_count = irc_flags.sum(axis=1)

irc_remove_mask = (
    irc_count >= IRC_MIN_TARGETS
)

removed_by_irc = int(
    irc_remove_mask.sum()
)

irc_removed_indices = irc_count.index[
    irc_remove_mask
]

train_before_irc = len(train_df)

train_df = train_df.loc[
    ~irc_remove_mask
].copy()

print("\nIRC rows flagged in at least",
      IRC_MIN_TARGETS, "targets:",
      removed_by_irc)

print("Rows before IRC:", train_before_irc)
print("Rows after IRC:", len(train_df))

if len(train_df) < IRC_FOLDS * 10:
    raise ValueError(
        "بعد از IRC داده آموزشی کافی باقی نمانده است."
    )


# ============================================================
# 10. آموزش نهایی شش مدل و پیش‌بینی سه ماه اخیر
# ============================================================

print("\n[7/9] Training final models and predicting...")

recent_df = df.loc[recent_indices].copy()

model_status = []
prediction_columns = []

for target in targets:

    model_start = time.perf_counter()

    print("\nFinal model target:", target)

    pred_col = f"predicted {target}"
    prediction_columns.append(pred_col)

    # ستون پیش‌بینی فقط برای داده سه ماه اخیر خروجی دارد.
    recent_df[pred_col] = np.nan

    input_features = [
        col for col in features
        if col != target
    ]

    # -----------------------------
    # داده آموزشی نهایی
    # -----------------------------

    X_train_all = train_df[input_features].copy()

    y_train_all = pd.to_numeric(
        train_df[target],
        errors="coerce"
    )

    X_train_all = X_train_all.replace(
        [np.inf, -np.inf],
        np.nan
    )

    valid_target_mask = y_train_all.notna()

    X_train_all = X_train_all.loc[
        valid_target_mask
    ]

    y_train = y_train_all.loc[
        valid_target_mask
    ]

    usable_features = [
        col for col in input_features
        if X_train_all[col].notna().any()
    ]

    if not usable_features:
        raise ValueError(
            f"هیچ ورودی معتبری برای مدل نهایی {target} وجود ندارد."
        )

    X_train = X_train_all[usable_features].copy()

    # میانه فقط از مجموعه آموزش نهایی
    medians = X_train.median().fillna(0)

    X_train = X_train.fillna(medians)

    if len(X_train) < 50:
        raise ValueError(
            f"نمونه آموزشی مدل {target} کمتر از 50 است."
        )

    # -----------------------------
    # ورودی سه ماه اخیر
    # -----------------------------

    X_predict = recent_df[usable_features].copy()

    X_predict = X_predict.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # پرکردن مقادیر گمشده با میانه آموزش
    X_predict = X_predict.fillna(medians)

    if X_predict.isna().any().any():
        raise ValueError(
            f"ورودی پیش‌بینی {target} دارای مقدار گمشده است."
        )

    # -----------------------------
    # آموزش مدل نهایی
    # -----------------------------

    final_model = make_xgb()

    final_model.fit(
        X_train,
        y_train,
        verbose=False
    )

    # -----------------------------
    # تولید پیش‌بینی
    # -----------------------------

    predictions = final_model.predict(
        X_predict
    )

    if not np.isfinite(predictions).all():
        raise RuntimeError(
            f"پیش‌بینی مدل {target} دارای مقدار نامعتبر است."
        )

    if len(predictions) != len(recent_df):
        raise RuntimeError(
            f"تعداد پیش‌بینی‌های {target} با تعداد ردیف‌ها برابر نیست."
        )

    recent_df[pred_col] = predictions

    written_count = int(
        recent_df[pred_col].notna().sum()
    )

    elapsed = time.perf_counter() - model_start

    print("Final training rows:", len(X_train))
    print("Input features:", len(usable_features))
    print("Recent predictions:", written_count)
    print("Final model time (sec):", round(elapsed, 2))

    if written_count == 0:
        raise RuntimeError(
            f"ستون {pred_col} خالی مانده است."
        )

    model_status.append({
        "Target": target,
        "Input features": len(usable_features),
        "Training rows": len(X_train),
        "Recent rows predicted": written_count,
        "Final model time (sec)": round(elapsed, 2),
        "Status": "OK"
    })


# ============================================================
# 11. کنترل خروجی: فقط سه ماه اخیر
# ============================================================

print("\n[8/9] Validating output...")

recent_df.sort_values(
    DATE_COLUMN,
    inplace=True
)

recent_df.reset_index(drop=True, inplace=True)

if len(recent_df) != len(recent_indices):
    raise RuntimeError(
        "تعداد ردیف‌های خروجی با تعداد ردیف‌های سه ماه اخیر برابر نیست."
    )

for pred_col in prediction_columns:

    count = int(
        recent_df[pred_col].notna().sum()
    )

    print(f"{pred_col}: {count} predictions")

    if count == 0:
        raise RuntimeError(
            f"ستون {pred_col} خالی است؛ خروجی ذخیره نمی‌شود."
        )


# ============================================================
# 12. گزارش پالایش و ذخیره فایل Excel
# ============================================================

print("\n[9/9] Saving output...")

model_status_df = pd.DataFrame(
    model_status
)

irc_details_df = pd.DataFrame(
    irc_details
)

summary_df = pd.DataFrame({
    "Metric": [
        "Input rows",
        "Latest input date",
        "Three-month cutoff",
        "Historical rows",
        "Rows removed by power filter",
        "Training rows after power filter",
        "Total IQR replacements",
        "Rows removed by Isolation Forest",
        "Rows removed by IRC",
        "Final training rows",
        "Recent rows in output",
        "Recent rows predicted per target",
        "Total elapsed time (sec)"
    ],
    "Value": [
        input_row_count,
        str(max_date),
        str(cutoff_date),
        len(old_indices),
        removed_by_power,
        before_power_filter - removed_by_power,
        total_iqr_replacements,
        removed_by_iso,
        removed_by_irc,
        len(train_df),
        len(recent_df),
        len(recent_df),
        round(time.perf_counter() - START_TIME, 2)
    ]
})

# ============================================================
# 13. ارزیابی رگرسیون روی سه ماه اخیر
#     این داده‌ها در آموزش مدل نهایی استفاده نشده‌اند.
# ============================================================

print("\nCalculating regression evaluation metrics...")

evaluation_rows = []
evaluation_guide_rows = [
    {"Metric": "Valid evaluation rows", "Description": "تعداد ردیف‌هایی که مقدار واقعی و پیش‌بینی هر دو معتبر هستند.", "Preferred direction": "بیشتر، با توجه به داده موجود"},
    {"Metric": "MAE", "Description": "میانگین قدر مطلق اختلاف مقدار واقعی و پیش‌بینی؛ هم‌واحد با سنسور.", "Preferred direction": "کمتر بهتر است؛ صفر بهترین حالت است."},
    {"Metric": "MSE", "Description": "میانگین مربع خطا؛ خطاهای بزرگ را شدیدتر جریمه می‌کند.", "Preferred direction": "کمتر بهتر است؛ صفر بهترین حالت است."},
    {"Metric": "RMSE", "Description": "ریشه MSE؛ هم‌واحد با سنسور و حساس به خطاهای بزرگ.", "Preferred direction": "کمتر بهتر است؛ صفر بهترین حالت است."},
    {"Metric": "R2", "Description": "ضریب تعیین؛ عملکرد را نسبت به پیش‌بینی میانگین مقدار واقعی در مجموعه ارزیابی می‌سنجد.", "Preferred direction": "معمولاً نزدیک‌تر به 1 بهتر است؛ ممکن است منفی شود."},
    {"Metric": "Explained Variance", "Description": "میزان تغییرپذیری مقادیر واقعی که پیش‌بینی‌ها توضیح می‌دهند.", "Preferred direction": "معمولاً نزدیک‌تر به 1 بهتر است."},
    {"Metric": "Median Absolute Error", "Description": "میانه قدر مطلق خطا؛ نسبت به خطاهای بسیار بزرگ مقاوم‌تر از MAE است.", "Preferred direction": "کمتر بهتر است؛ صفر بهترین حالت است."},
    {"Metric": "Mean Error (Predicted - Actual)", "Description": "میانگین پیش‌بینی منهای مقدار واقعی؛ علامت مثبت نشان‌دهنده بیش‌برآوردی و منفی نشان‌دهنده کم‌برآوردی است.", "Preferred direction": "نزدیک صفر بهتر است."},
    {"Metric": "MAPE (%)", "Description": "میانگین درصد قدر مطلق خطا؛ ردیف‌هایی که مقدار واقعی صفر است از محاسبه حذف می‌شوند.", "Preferred direction": "کمتر بهتر است؛ برای مقادیر واقعی نزدیک صفر با احتیاط تفسیر شود."},
    {"Metric": "sMAPE (%)", "Description": "معیار درصدی متقارن خطا؛ در صورت صفر بودن هم‌زمان مقدار واقعی و پیش‌بینی، آن ردیف در میانگین لحاظ نمی‌شود.", "Preferred direction": "کمتر بهتر است."},
    {"Metric": "Baseline RMSE", "Description": "RMSE یک مدل ساده که برای تمام ردیف‌های ارزیابی، میانگین هدف در داده آموزشی نهایی را پیش‌بینی می‌کند.", "Preferred direction": "برای مقایسه؛ هرچه کمتر باشد بهتر است."},
    {"Metric": "RMSE improvement vs baseline (%)", "Description": "درصد کاهش RMSE نسبت به خط مبنای میانگین آموزشی؛ مقدار مثبت یعنی XGBoost بهتر از خط مبناست.", "Preferred direction": "بیشتر بهتر است؛ مقدار منفی یعنی مدل از خط مبنا ضعیف‌تر است."},
]

for target in targets:
    pred_col = f"predicted {target}"

    actual_values = pd.to_numeric(recent_df[target], errors="coerce")
    predicted_values = pd.to_numeric(recent_df[pred_col], errors="coerce")

    valid_mask = (
        actual_values.notna()
        & predicted_values.notna()
        & np.isfinite(actual_values)
        & np.isfinite(predicted_values)
    )

    y_true = actual_values.loc[valid_mask].to_numpy(dtype=float)
    y_pred = predicted_values.loc[valid_mask].to_numpy(dtype=float)

    n_eval = len(y_true)

    if n_eval == 0:
        evaluation_rows.append({
            "Target": target,
            "Valid evaluation rows": 0,
            "MAE": np.nan,
            "MSE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "Explained Variance": np.nan,
            "Median Absolute Error": np.nan,
            "Mean Error (Predicted - Actual)": np.nan,
            "MAPE (%)": np.nan,
            "sMAPE (%)": np.nan,
            "Baseline RMSE": np.nan,
            "RMSE improvement vs baseline (%)": np.nan,
            "Evaluation status": "No valid actual/predicted pairs"
        })
        continue

    errors = y_pred - y_true
    absolute_errors = np.abs(errors)
    mse_value = mean_squared_error(y_true, y_pred)
    rmse_value = float(np.sqrt(mse_value))

    # R2 و Explained Variance برای تعداد کمتر از 2 نمونه قابل اتکا نیستند.
    if n_eval >= 2:
        r2_value = r2_score(y_true, y_pred)
        explained_variance_value = explained_variance_score(y_true, y_pred)
    else:
        r2_value = np.nan
        explained_variance_value = np.nan

    # MAPE: مقدار واقعی صفر از مخرج حذف می‌شود.
    nonzero_actual = np.abs(y_true) > 1e-12
    if nonzero_actual.any():
        mape_value = float(
            np.mean(np.abs((y_true[nonzero_actual] - y_pred[nonzero_actual]) / y_true[nonzero_actual])) * 100
        )
    else:
        mape_value = np.nan

    # sMAPE: جفت‌هایی که مقدار واقعی و پیش‌بینی هر دو صفرند، کنار گذاشته می‌شوند.
    smape_denominator = np.abs(y_true) + np.abs(y_pred)
    valid_smape = smape_denominator > 1e-12
    if valid_smape.any():
        smape_value = float(
            np.mean(2.0 * absolute_errors[valid_smape] / smape_denominator[valid_smape]) * 100
        )
    else:
        smape_value = np.nan

    # خط مبنا: پیش‌بینی ثابت برابر میانگین هدف در آموزش نهایی همان مدل.
    train_actual = pd.to_numeric(train_df[target], errors="coerce").dropna()
    if len(train_actual) > 0:
        baseline_value = float(train_actual.mean())
        baseline_predictions = np.full(n_eval, baseline_value, dtype=float)
        baseline_rmse = float(np.sqrt(mean_squared_error(y_true, baseline_predictions)))
        if baseline_rmse > 1e-12:
            rmse_improvement = float((baseline_rmse - rmse_value) / baseline_rmse * 100)
        else:
            rmse_improvement = np.nan
    else:
        baseline_rmse = np.nan
        rmse_improvement = np.nan

    evaluation_rows.append({
        "Target": target,
        "Valid evaluation rows": n_eval,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MSE": float(mse_value),
        "RMSE": rmse_value,
        "R2": float(r2_value) if pd.notna(r2_value) else np.nan,
        "Explained Variance": float(explained_variance_value) if pd.notna(explained_variance_value) else np.nan,
        "Median Absolute Error": float(median_absolute_error(y_true, y_pred)),
        "Mean Error (Predicted - Actual)": float(np.mean(errors)),
        "MAPE (%)": mape_value,
        "sMAPE (%)": smape_value,
        "Baseline RMSE": baseline_rmse,
        "RMSE improvement vs baseline (%)": rmse_improvement,
        "Evaluation status": "OK" if n_eval >= 2 else "Only one valid pair; R2/Explained Variance unavailable"
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_guide_df = pd.DataFrame(evaluation_guide_rows)

print("\nRegression evaluation:")
print(evaluation_df.to_string(index=False))

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    recent_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    model_status_df.to_excel(
        writer,
        sheet_name="Model_Status",
        index=False
    )

    irc_details_df.to_excel(
        writer,
        sheet_name="IRC_Details",
        index=False
    )

    evaluation_df.to_excel(
        writer,
        sheet_name="Regression_Evaluation",
        index=False
    )

    evaluation_guide_df.to_excel(
        writer,
        sheet_name="Evaluation_Guide",
        index=False
    )

print("\n" + "=" * 70)
print("PROCESS COMPLETED")
print("=" * 70)
print("Output file:", output_file)
print("Output rows:", len(recent_df))
print("Final training rows:", len(train_df))
print("Rows removed by Isolation Forest:", removed_by_iso)
print("Rows removed by IRC:", removed_by_irc)
print(
    "Total elapsed time (sec):",
    round(time.perf_counter() - START_TIME, 2)
)
print("\nModel status:")
print(model_status_df.to_string(index=False))
print("=" * 70)
print("\n" + "=" * 70)
print("FINAL TRAINING DATA COUNT FOR EACH MODEL")
print("=" * 70)

for _, row in model_status_df.iterrows():
    print(
        f"Target: {row['Target']} | "
        f"Training samples: {row['Training rows']:,} | "
        f"Predictions: {row['Recent rows predicted']:,}"
    )

print("-" * 70)
print(f"Final cleaned training rows: {len(train_df):,}")
print(f"Recent output rows: {len(recent_df):,}")
print("=" * 70)
print("\nRegression evaluation sheet: Regression_Evaluation")
print("Evaluation guide sheet: Evaluation_Guide")



[1/9] Reading input file...
Input rows: 12358

[2/9] Defining training and prediction periods...
Latest date: 2026-10-27 12:40:41
Three-month cutoff: 2026-07-27 12:40:41
Historical rows: 11816
Recent output rows: 542

[3/9] Applying operational filter...
Historical rows before power filter: 11816
Rows removed by power filter: 901
Training rows after power filter: 10915

[4/9] Applying IQR cleaning...
IQR | ACTIVE POWER: 33 replacements
IQR | REACTIVE POWER: 27 replacements
IQR | GEN BRG VIB (COMP.) X: 458 replacements
IQR | GEN BRG VIB (COMP.) Y: 264 replacements
IQR | GEN BRG VIB (EXE SIDE) X: 2569 replacements
IQR | GEN BRG VIB (EXE SIDE) Y: 53 replacements
IQR | EXE. BRG VIB X: 1 replacements
IQR | EXE. BRG VIB Y: 94 replacements
IQR | GEN.BRG.DR.TEMP(COMP.SIDE): 1189 replacements
IQR | GEN.BRG.DR.TEMP(EXC.SIDE): 1272 replacements
IQR | EXC.BRG.DR.TEMP.: 360 replacements
IQR | AMBIENT TEMP: 4 replacements
Total IQR replacements: 6324

[5/9] Applying Isolation Forest...
Rows removed